# Pantanal BirdCLEF 2026 — Data Processing & Two-Phase Training

**Phase 1**: Pre-train on short bird clips (XC / iNat) — head only, backbone frozen  
**Phase 2**: Fine-tune on labeled soundscape segments — full model, lower LR

In [6]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

SEED = 42

bird_audio_dir = "/home/users/ss1482/sangcs372final/Finalproject/birdtrain_wav"
soundscape_dir = "/home/users/ss1482/sangcs372final/Finalproject/soundtrain"

# =========================
# 1. LOAD TRAIN AUDIO METADATA
# =========================
df = pd.read_csv("train.csv")

# Strip the subdirectory prefix, then convert extension
df["filename"] = df["filename"].apply(lambda x: os.path.basename(x))  # '1161364/iNat1216197.ogg' → 'iNat1216197.ogg'
df["filename"] = df["filename"].str.replace(".ogg", ".wav", regex=False)

df["filepath"] = df["filename"].apply(lambda x: os.path.join(bird_audio_dir, x))
df = df[df["filepath"].apply(os.path.exists)].reset_index(drop=True)
print("Files found:", len(df))
# =========================
# 2. LOAD SOUNDSCAPE LABELS
# =========================
ss_df = pd.read_csv("train_soundscapes_labels.csv")

# Convert filenames
ss_df["filename"] = ss_df["filename"].str.replace(".ogg", ".wav", regex=False)

# Path to soundscape wavs
soundscape_dir = "/home/users/ss1482/sangcs372final/Finalproject/soundtrain"

# Build filepaths
ss_df["filepath"] = ss_df["filename"].apply(lambda x: os.path.join(soundscape_dir, x))

# Remove missing files
ss_df = ss_df[ss_df["filepath"].apply(os.path.exists)].reset_index(drop=True)

print("Soundscape segments:", len(ss_df))


# =========================
# 3. LOAD TAXONOMY
# =========================
taxonomy = pd.read_csv("taxonomy.csv")
labels = taxonomy["primary_label"].values
label_to_idx = {label: i for i, label in enumerate(labels)}
idx_to_label = {i: label for label, i in label_to_idx.items()}
NUM_CLASSES = len(labels)
print("Total classes:", NUM_CLASSES)


# =========================
# 4. SPLIT BIRD CLIPS  (stratified by species)
# =========================
# Species with only 1 sample can't be stratified — pull them out first
counts = df["primary_label"].value_counts()
rare   = set(counts[counts < 2].index)

df_rare     = df[df["primary_label"].isin(rare)]
df_stratify = df[~df["primary_label"].isin(rare)]

bird_train_df, bird_val_df = train_test_split(
    df_stratify,
    test_size=0.2,
    stratify=df_stratify["primary_label"],
    random_state=SEED
)

# Rare species go entirely to train (can't validate what you barely have)
bird_train_df = pd.concat([bird_train_df, df_rare]).reset_index(drop=True)
bird_val_df   = bird_val_df.reset_index(drop=True)

print(f"\nBird clips  → train: {len(bird_train_df)}  val: {len(bird_val_df)}")


# =========================
# 5. SPLIT SOUNDSCAPES  (split by FILE, not by segment — avoids leakage)
# =========================
# Adjacent 5-second segments from the same file are highly correlated,
# so we group all segments from a file together before splitting.
unique_files = ss_df["filename"].unique()

ss_train_files, ss_val_files = train_test_split(
    unique_files,
    test_size=0.2,
    random_state=SEED
)

ss_train_df = ss_df[ss_df["filename"].isin(ss_train_files)].reset_index(drop=True)
ss_val_df   = ss_df[ss_df["filename"].isin(ss_val_files)].reset_index(drop=True)

print(f"Soundscapes → train files: {len(ss_train_files)}  val files: {len(ss_val_files)}")
print(f"             train segs:  {len(ss_train_df)}  val segs:  {len(ss_val_df)}")

print("\nUnique train_audio species:", df["primary_label"].nunique())
print("Species missing from train_audio:", NUM_CLASSES - df["primary_label"].nunique())

Files found: 12729
Soundscape segments: 1478
Total classes: 234

Bird clips  → train: 10183  val: 2546
Soundscapes → train files: 52  val files: 14
             train segs:  1160  val segs:  318

Unique train_audio species: 80
Species missing from train_audio: 154


In [7]:
import numpy as np
import soundfile as sf
import librosa

TARGET_SR = 32000

def load_audio_segment(filepath, target_len, start_sec=None):
    try:
        audio, sr = sf.read(filepath)
    except:
        return np.zeros(target_len, dtype=np.float32)

    if audio.ndim == 2:
        audio = np.mean(audio, axis=1)

    if sr != TARGET_SR:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=TARGET_SR)

    audio = audio.astype(np.float32)

    if start_sec is not None:
        start = int(start_sec * TARGET_SR)
        end   = start + target_len
        audio = audio[start:end]

    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]

    # ── Normalize so the backbone gets consistent input energy ──
    max_val = np.abs(audio).max()
    if max_val > 0:
        audio = audio / max_val          # peak normalize to [-1, 1]

    return audio


In [8]:
import torch
from torch.utils.data import Dataset

class BaseAudioDataset(Dataset):
    def __init__(self, df, label_to_idx, clip_duration, multi_label=False, augment=False):
        self.df           = df.reset_index(drop=True)
        self.label_to_idx = label_to_idx
        self.clip_len     = TARGET_SR * clip_duration
        self.multi_label  = multi_label
        self.augment      = augment

    def __len__(self):
        return len(self.df)

    def get_audio(self, row):
        return load_audio_segment(row["filepath"], self.clip_len)

    def get_label(self, row):
        label = torch.zeros(len(self.label_to_idx))
        if self.multi_label:
            for sp in str(row["primary_label"]).split(";"):
                if sp in self.label_to_idx:
                    label[self.label_to_idx[sp]] = 1.0
        else:
            if row["primary_label"] in self.label_to_idx:
                label[self.label_to_idx[row["primary_label"]]] = 1.0
        return label

    def _maybe_augment(self, audio):
        """Light waveform augmentation — only applied during training."""
        if not self.augment:
            return audio
        # Random gain
        gain  = np.random.uniform(0.6, 1.4)
        audio = audio * gain
        # Gaussian noise
        if np.random.rand() < 0.5:
            noise = np.random.randn(len(audio)).astype(np.float32)
            audio = audio + noise * np.random.uniform(0.001, 0.01)
        return np.clip(audio, -1.0, 1.0)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        audio = self.get_audio(row)
        audio = self._maybe_augment(audio)
        audio = torch.tensor(audio) 
        label = self.get_label(row)
        return audio, label

In [9]:
class BirdDataset(BaseAudioDataset):
    """Short XC/iNat clips — single primary label, 10-second clips."""
    def __init__(self, df, label_to_idx, augment=False):
        super().__init__(df, label_to_idx, clip_duration=10,
                         multi_label=False, augment=augment)

In [10]:
class SoundscapeDataset(BaseAudioDataset):
    """Labeled soundscape segments — multi-label, 5-second clips."""
    def __init__(self, df, label_to_idx, augment=False):
        super().__init__(df, label_to_idx, clip_duration=5,
                         multi_label=True, augment=augment)

    def get_audio(self, row):
        return load_audio_segment(
            row["filepath"],
            self.clip_len,
            start_sec=row["start"]
        )

In [13]:
from torch.utils.data import DataLoader
batch_size = 16
batch_size = 16
# ── Bird clip datasets (augment train only) ──────────────────
bird_train_ds = BirdDataset(bird_train_df, label_to_idx, augment=True)
bird_val_ds   = BirdDataset(bird_val_df,   label_to_idx, augment=False)

bird_train_loader = DataLoader(bird_train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=1, pin_memory=batch_size, drop_last=True)
bird_val_loader   = DataLoader(bird_val_ds,   batch_size=64, shuffle=False,
                               num_workers=1, pin_memory=True)

# ── Soundscape datasets ──────────────────────────────────────
ss_train_ds = SoundscapeDataset(ss_train_df, label_to_idx, augment=True)
ss_val_ds   = SoundscapeDataset(ss_val_df,   label_to_idx, augment=False)

ss_train_loader = DataLoader(ss_train_ds, batch_size=16, shuffle=True,
                             num_workers=1, pin_memory=True, drop_last=True)
ss_val_loader   = DataLoader(ss_val_ds,   batch_size=16, shuffle=False,
                             num_workers=1, pin_memory=True)

print(f"Bird    — train batches: {len(bird_train_loader)}  val batches: {len(bird_val_loader)}")
print(f"Soundsc — train batches: {len(ss_train_loader)}  val batches: {len(ss_val_loader)}")

Bird    — train batches: 636  val batches: 40
Soundsc — train batches: 72  val batches: 20


In [14]:
import sys
sys.path.append("/home/users/ss1482/sangcs372final/audioset_tagging_cnn/pytorch")

from pytorch.models import Cnn14

In [15]:
model = Cnn14(
    sample_rate=32000,
    window_size=1024,
    hop_size=320,
    mel_bins=64,
    fmin=50,
    fmax=14000,
    classes_num=527   # original AudioSet head — will be replaced below
)

In [16]:
checkpoint = torch.load("Cnn14_mAP=0.431.pth", map_location="cpu")
model.load_state_dict(checkpoint["model"], strict=False)
print("Pretrained weights loaded.")

Pretrained weights loaded.


In [19]:
# Replace AudioSet head (527 classes) with our Pantanal head (234 classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.fc_audioset = torch.nn.Sequential(
    torch.nn.Linear(2048, 512),
    torch.nn.BatchNorm1d(512),   # normalizes the sparse fc1 output
    torch.nn.GELU(),             # GELU doesn't hard-zero negatives like ReLU does
    torch.nn.Dropout(0.3),
    torch.nn.Linear(512, NUM_CLASSES)
).to(device)


model  = model.to(device)
print("Model ready on:", device)

Model ready on: cuda


In [20]:
def freeze_backbone_partial(model):
    for name, param in model.named_parameters():
        param.requires_grad = False

    for name, param in model.named_parameters():
        if any(x in name for x in ["conv_block4", "conv_block5", "conv_block6", "fc1", "fc_audioset"]):
            param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Partial freeze. Trainable params: {trainable:,}")
def unfreeze_all(model):
    """Unfreeze every layer for full fine-tuning."""
    for param in model.parameters():
        param.requires_grad = True
    total = sum(p.numel() for p in model.parameters())
    print(f"All layers unfrozen. Total trainable params: {total:,}")

In [21]:
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

def run_validation(model, loader, criterion, device):
    """Returns average val loss and macro ROC-AUC."""
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for audio, label in loader:
            audio  = audio.to(device)
            label  = label.to(device)
            output = model(audio)
            # CNN14 returns a dict; pull the clipwise logits
            logits = output["clipwise_output"] if isinstance(output, dict) else output
            loss   = criterion(logits, label)
            total_loss += loss.item()
            all_preds.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(label.cpu().numpy())

    all_preds  = np.concatenate(all_preds,  axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # Only score classes that actually appear in this val split
    valid_cols = all_labels.sum(axis=0) > 0
    try:
        auc = roc_auc_score(all_labels[:, valid_cols], all_preds[:, valid_cols],
                            average="macro")
    except Exception:
        auc = float("nan")

    return total_loss / len(loader), auc

## Phase 1 — Pre-train on Bird Clips (backbone frozen)

Only the new 234-class head is trained here.  
The CNN14 backbone keeps its AudioSet weights and learns nothing yet — this prevents catastrophic forgetting while the head finds reasonable initialisation.

In [24]:
# Run ONE forward+backward pass and check if gradients are actually flowing
model.train()
audio, label = next(iter(bird_train_loader))
audio = audio.to(device)
label = label.to(device)

optimizer.zero_grad()
output = model(audio)
logits = output["clipwise_output"] if isinstance(output, dict) else output
loss   = criterion(logits, label)
loss.backward()

# Check gradient magnitudes on the head and a mid-level layer
for name, param in model.named_parameters():
    if param.grad is not None and param.grad.abs().max() > 0:
        print(f"{name:50s}  grad_max={param.grad.abs().max():.6f}")

conv_block4.conv1.weight                            grad_max=0.000157
conv_block4.conv2.weight                            grad_max=0.000116
conv_block4.bn1.weight                              grad_max=0.000749
conv_block4.bn1.bias                                grad_max=0.000419
conv_block4.bn2.weight                              grad_max=0.000723
conv_block4.bn2.bias                                grad_max=0.000539
conv_block5.conv1.weight                            grad_max=0.000265
conv_block5.conv2.weight                            grad_max=0.000120
conv_block5.bn1.weight                              grad_max=0.001440
conv_block5.bn1.bias                                grad_max=0.000286
conv_block5.bn2.weight                              grad_max=0.000950
conv_block5.bn2.bias                                grad_max=0.000306
conv_block6.conv1.weight                            grad_max=0.000156
conv_block6.conv2.weight                            grad_max=0.000180
conv_block6.bn1.weig

In [31]:
import torch.nn as nn
import torch.nn.functional as F

class Cnn14Pantanal(nn.Module):
    def __init__(self, base_model, num_classes):
        super().__init__()
        # Copy all layers from the pretrained model
        self.spectrogram_extractor = base_model.spectrogram_extractor
        self.logmel_extractor      = base_model.logmel_extractor
        self.spec_augmenter        = base_model.spec_augmenter
        self.bn0         = base_model.bn0
        self.conv_block1 = base_model.conv_block1
        self.conv_block2 = base_model.conv_block2
        self.conv_block3 = base_model.conv_block3
        self.conv_block4 = base_model.conv_block4
        self.conv_block5 = base_model.conv_block5
        self.conv_block6 = base_model.conv_block6
        self.fc1         = base_model.fc1   # keep pretrained weights
        self.gelu        = nn.GELU()        # replaces F.relu_

        # New head for 234 classes
        self.fc_audioset = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, input):
        x = self.spectrogram_extractor(input)
        x = self.logmel_extractor(x)

        x = x.transpose(1, 3)
        x = self.bn0(x)
        x = x.transpose(1, 3)

        if self.training:
            x = self.spec_augmenter(x)

        x = self.conv_block1(x, pool_size=(2, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block2(x, pool_size=(2, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block3(x, pool_size=(2, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block4(x, pool_size=(2, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block5(x, pool_size=(2, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block6(x, pool_size=(1, 1), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)

        x = torch.mean(x, dim=3)
        (x1, _) = torch.max(x, dim=2)
        x2 = torch.mean(x, dim=2)
        x = x1 + x2

        x = F.dropout(x, p=0.5, training=self.training)
        x = self.gelu(self.fc1(x))          # GELU instead of F.relu_
        x = F.dropout(x, p=0.5, training=self.training)

        logits = self.fc_audioset(x)        # raw logits — no sigmoid here

        return {"clipwise_output": logits, "embedding": x}


# ── Wrap the existing model ──────────────────────────────────
model = Cnn14Pantanal(model, NUM_CLASSES).to(device)

# ── Verify fc1 zeros are gone ────────────────────────────────
activations = {}
def hook_fn(module, input, output):
    activations["fc1"] = output.detach()

hook = model.fc1.register_forward_hook(hook_fn)
audio, label = next(iter(bird_train_loader))
with torch.no_grad():
    model(audio.to(device))
hook.remove()

print("fc1 output mean:", activations["fc1"].mean().item())
print("fc1 output std: ", activations["fc1"].std().item())
print("fc1 zeros:      ", (activations["fc1"] == 0).float().mean().item())

fc1 output mean: -0.060371194034814835
fc1 output std:  0.3049980103969574
fc1 zeros:       0.0142822265625


In [41]:
import os
import torch.optim as optim

# ── Hyperparameters ──────────────────────────────────────────
PHASE1_EPOCHS  = 10
WEIGHT_DECAY   = 1e-2
BATCH_SIZE     = 8
WARMUP_EPOCHS  = 0
PATIENCE       = 3
MIN_DELTA      = 1e-4
RESUME         = True   # ← NEW

# ── Early stopping state ─────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience=PATIENCE, min_delta=MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_auc   = 0.0
        self.should_stop = False

    def step(self, val_auc):
        if val_auc > self.best_auc + self.min_delta:
            self.best_auc  = val_auc
            self.counter   = 0
        else:
            self.counter  += 1
            print(f"  ↳ No improvement for {self.counter}/{self.patience} epochs")
            if self.counter >= self.patience:
                self.should_stop = True
                print("  ✗ Early stopping triggered.")

# Freeze backbone
freeze_backbone_partial(model)

criterion = torch.nn.BCEWithLogitsLoss()

optimizer = optim.AdamW([
    {"params": [p for n,p in model.named_parameters() if "conv_block4" in n], "lr": 1e-5},
    {"params": [p for n,p in model.named_parameters() if "conv_block5" in n], "lr": 5e-5},
    {"params": [p for n,p in model.named_parameters() if "conv_block6" in n], "lr": 1e-4},
    {"params": [p for n,p in model.named_parameters() if "fc1" in n],         "lr": 2e-4},
    {"params": [p for n,p in model.named_parameters() if "fc_audioset" in n], "lr": 5e-4},
], weight_decay=WEIGHT_DECAY)

def get_lr_scale(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / max(1, WARMUP_EPOCHS)
    progress = (epoch - WARMUP_EPOCHS) / max(1, PHASE1_EPOCHS - WARMUP_EPOCHS)
    return 0.1 + 0.9 * 0.5 * (1 + torch.cos(torch.tensor(3.14159 * progress)).item())

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr_scale)

early_stopper = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA)

# ── Resume logic ─────────────────────────────────────────────
start_epoch = 1
best_phase1_auc = 0.0

if RESUME and os.path.exists("best_phase1.pth"):
    ckpt = torch.load("best_phase1.pth", map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])

    if "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])

    start_epoch = ckpt["epoch"] + 1
    best_phase1_auc = ckpt["val_auc"]

    print(f"Resuming from epoch {start_epoch-1}, best AUC {best_phase1_auc:.4f}")

# ── Training loop ────────────────────────────────────────────
for epoch in range(start_epoch, PHASE1_EPOCHS + 1):

    model.train()
    running_loss = 0.0

    for batch_idx, (audio, label) in enumerate(bird_train_loader):
        audio = audio.to(device)
        label = label.to(device)

        optimizer.zero_grad()
        output = model(audio)
        logits = output["clipwise_output"] if isinstance(output, dict) else output
        loss   = criterion(logits, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        if (batch_idx + 1) % 50 == 0:
            current_lr = optimizer.param_groups[0]["lr"]
            print(f"  [P1 E{epoch} step {batch_idx+1}] "
                  f"loss: {running_loss/(batch_idx+1):.4f}  lr: {current_lr:.2e}")

    scheduler.step()

    # ── validate ─────────────────────────────────────────────
    val_loss, val_auc = run_validation(model, bird_val_loader, criterion, device)
    current_lr = optimizer.param_groups[0]["lr"]

    print(f"Epoch {epoch}/{PHASE1_EPOCHS}  "
          f"train_loss={running_loss/len(bird_train_loader):.4f}  "
          f"val_loss={val_loss:.4f}  val_auc={val_auc:.4f}  "
          f"lr={current_lr:.2e}")

    # ── save latest (NEW, optional but useful) ────────────────
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "val_auc": val_auc,
        "batch_size": BATCH_SIZE,
    }, "latest_phase1.pth")

    # ── checkpoint if best ───────────────────────────────────
    if val_auc > best_phase1_auc:
        best_phase1_auc = val_auc
        torch.save({
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),  # ← added
            "val_auc": val_auc,
            "batch_size": BATCH_SIZE,
        }, "best_phase1.pth")
        print("  ✓ Saved best Phase 1 checkpoint")

    # ── early stopping ───────────────────────────────────────
    early_stopper.step(val_auc)
    if early_stopper.should_stop:
        print(f"\nStopped early at epoch {epoch}. "
              f"Best val AUC: {best_phase1_auc:.4f}")
        break

else:
    print(f"\nPhase 1 complete. Best val AUC: {best_phase1_auc:.4f}")

# ── safe reload ──────────────────────────────────────────────
if os.path.exists("best_phase1.pth"):
    print("Reloading best Phase 1 weights...")
    model.load_state_dict(torch.load("best_phase1.pth")["model"])
else:
    print("No checkpoint found to reload.")

Partial freeze. Trainable params: 79,698,666
Resuming from epoch 7, best AUC 0.9786
  [P1 E8 step 50] loss: 0.0076  lr: 7.54e-06
  [P1 E8 step 100] loss: 0.0076  lr: 7.54e-06
  [P1 E8 step 150] loss: 0.0076  lr: 7.54e-06
  [P1 E8 step 200] loss: 0.0077  lr: 7.54e-06
  [P1 E8 step 250] loss: 0.0077  lr: 7.54e-06
  [P1 E8 step 300] loss: 0.0078  lr: 7.54e-06
  [P1 E8 step 350] loss: 0.0077  lr: 7.54e-06
  [P1 E8 step 400] loss: 0.0077  lr: 7.54e-06
  [P1 E8 step 450] loss: 0.0077  lr: 7.54e-06
  [P1 E8 step 500] loss: 0.0077  lr: 7.54e-06
  [P1 E8 step 550] loss: 0.0077  lr: 7.54e-06
  [P1 E8 step 600] loss: 0.0076  lr: 7.54e-06
Epoch 8/10  train_loss=0.0076  val_loss=0.0071  val_auc=0.9717  lr=9.78e-06
  [P1 E9 step 50] loss: 0.0073  lr: 9.78e-06
  [P1 E9 step 100] loss: 0.0071  lr: 9.78e-06
  [P1 E9 step 150] loss: 0.0070  lr: 9.78e-06
  [P1 E9 step 200] loss: 0.0070  lr: 9.78e-06
  [P1 E9 step 250] loss: 0.0071  lr: 9.78e-06
  [P1 E9 step 300] loss: 0.0072  lr: 9.78e-06
  [P1 E9 step 

In [36]:
ckpt = torch.load("best_phase1.pth", map_location=device)

model.load_state_dict(ckpt["model"])
optimizer.load_state_dict(ckpt["optimizer"])

start_epoch = ckpt["epoch"]
best_phase1_auc = ckpt["val_auc"]

## Phase 2 — Fine-tune on Labeled Soundscapes (all layers unfrozen)

Now we unfreeze the backbone and train on the real field recordings.  
We use **discriminative learning rates**: the backbone gets a much lower LR than the head to avoid destroying what it learned in Phase 1.

In [40]:
for i in range(10):
    row = ss_df.iloc[i]
    print(row)

filename                 BC2026_Train_0039_S22_20211231_201500.wav
start                                                     00:00:00
end                                                       00:00:05
primary_label                       22961;23158;24321;517063;65380
filepath         /home/users/ss1482/sangcs372final/Finalproject...
Name: 0, dtype: object
filename                 BC2026_Train_0039_S22_20211231_201500.wav
start                                                     00:00:05
end                                                       00:00:10
primary_label                       22961;23158;24321;517063;65380
filepath         /home/users/ss1482/sangcs372final/Finalproject...
Name: 1, dtype: object
filename                 BC2026_Train_0039_S22_20211231_201500.wav
start                                                     00:00:10
end                                                       00:00:15
primary_label                       22961;23158;24321;517063;65380
filepath        

In [42]:
# ── Load best Phase 1 weights before unfreezing ──────────────
ckpt = torch.load("best_phase1.pth", map_location=device)
model.load_state_dict(ckpt["model"])

unfreeze_all(model)
# ── Config ───────────────────────────────────────────────────
PHASE2_EPOCHS   = 10
BACKBONE_LR     = 1e-5   # very low — backbone already has good features
HEAD_LR         = 1e-4   # higher — head still adapting

# Discriminative LRs: separate param groups for backbone vs head
head_params     = list(model.fc_audioset.parameters())
head_ids        = set(id(p) for p in head_params)
backbone_params = [p for p in model.parameters() if id(p) not in head_ids]

optimizer = optim.Adam([
    {"params": backbone_params, "lr": BACKBONE_LR},
    {"params": head_params,     "lr": HEAD_LR},
])

# Cosine LR scheduler over Phase 2
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=PHASE2_EPOCHS, eta_min=1e-7
)

best_phase2_auc = 0.0

for epoch in range(1, PHASE2_EPOCHS + 1):
    # ── train ────────────────────────────────────────────────
    model.train()
    running_loss = 0.0

    for batch_idx, (audio, label) in enumerate(ss_train_loader):
        audio = audio.to(device)
        label = label.to(device)

        optimizer.zero_grad()
        output = model(audio)
        logits = output["clipwise_output"] if isinstance(output, dict) else output
        loss   = criterion(logits, label)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if (batch_idx + 1) % 20 == 0:
            print(f"  [P2 E{epoch} step {batch_idx+1}] loss: {running_loss/(batch_idx+1):.4f}")

    scheduler.step()

    # ── validate ─────────────────────────────────────────────
    val_loss, val_auc = run_validation(model, ss_val_loader, criterion, device)
    current_backbone_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch}/{PHASE2_EPOCHS}  "
          f"train_loss={running_loss/len(ss_train_loader):.4f}  "
          f"val_loss={val_loss:.4f}  val_auc={val_auc:.4f}  "
          f"backbone_lr={current_backbone_lr:.2e}")

    if val_auc > best_phase2_auc:
        best_phase2_auc = val_auc
        torch.save(model.state_dict(), "best_phase2.pth")
        print("  ✓ Saved best Phase 2 checkpoint")

print(f"\nPhase 2 complete. Best val AUC: {best_phase2_auc:.4f}")
print("Final model saved to best_phase2.pth")

All layers unfrozen. Total trainable params: 81,927,402


ValueError: Caught ValueError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/users/ss1482/.local/lib/python3.11/site-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/users/ss1482/.local/lib/python3.11/site-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/users/ss1482/.local/lib/python3.11/site-packages/torch/utils/data/_utils/fetch.py", line 54, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_3535939/356462053.py", line 44, in __getitem__
    audio = self.get_audio(row)
            ^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_3535939/2955124301.py", line 8, in get_audio
    r

In [ ]:
# Quick sanity check — one batch through the final model
model.load_state_dict(torch.load("best_phase2.pth", map_location=device))
model.eval()

audio, label = next(iter(ss_val_loader))
audio = audio.to(device)

with torch.no_grad():
    output = model(audio)
    logits = output["clipwise_output"] if isinstance(output, dict) else output
    probs  = torch.sigmoid(logits)

print("Output shape:", probs.shape)   # should be (batch, 234)
print("Min prob:", probs.min().item(), "Max prob:", probs.max().item())